# TaxGPT — Episode 6: Layer Normalization and Residual Connections

Companion notebook to blog post *"Layer Normalization and Residual Connections: The Plumbing That Makes Deep Transformers Trainable (TaxGPT Episode 6)"*.

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 4.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

## 1. Layer normalization, built from scratch

In [2]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(emb_dim))
        self.beta = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * normalized + self.beta

ln = LayerNorm(768)
x = torch.randn(2, 5, 768) * 10 + 3  # deliberately off-center, large scale

out = ln(x)
print("before norm -> mean:", x.mean().item(), " std:", x.std().item())
print("after norm  -> per-token mean (should be ~0):", out.mean(dim=-1)[0,0].item())
print("after norm  -> per-token std  (should be ~1 before gamma/beta):",
      ((x - x.mean(dim=-1, keepdim=True)) / torch.sqrt(x.var(dim=-1, keepdim=True, unbiased=False) + 1e-5)).std(dim=-1)[0,0].item())

before norm -> mean: 2.936657190322876  std: 10.042180061340332
after norm  -> per-token mean (should be ~0): 1.3659398057086491e-08
after norm  -> per-token std  (should be ~1 before gamma/beta): 1.0006515979766846


## 2. Confirming it matches PyTorch's built-in LayerNorm

Sanity check the from-scratch version against `nn.LayerNorm` with matching gamma/beta.

In [3]:
torch_ln = nn.LayerNorm(768, eps=1e-5)
with torch.no_grad():
    torch_ln.weight.copy_(ln.gamma)
    torch_ln.bias.copy_(ln.beta)

out_custom = ln(x)
out_torch = torch_ln(x)
print("max abs difference vs nn.LayerNorm:", (out_custom - out_torch).abs().max().item())

max abs difference vs nn.LayerNorm: 4.76837158203125e-07


## 3. Residual connections: why the gradient path matters

Compare gradient magnitude reaching the *first* layer of a deep stack, with and without residual connections.

In [4]:
def deep_stack_no_residual(x, n_layers=20, dim=64):
    for _ in range(n_layers):
        w = torch.randn(dim, dim, requires_grad=False) * 0.9  # unnormalized scale -- grows variance when summed over `dim` inputs
        x = torch.tanh(x @ w)
    return x

def deep_stack_with_residual(x, n_layers=20, dim=64):
    for _ in range(n_layers):
        w = torch.randn(dim, dim, requires_grad=False) * 0.9
        x = x + torch.tanh(x @ w)   # residual connection
    return x

dim = 64
x_no_res = torch.randn(1, dim, requires_grad=True)
x_with_res = x_no_res.clone().detach().requires_grad_(True)

out_no_res = deep_stack_no_residual(x_no_res)
out_with_res = deep_stack_with_residual(x_with_res)

out_no_res.sum().backward()
out_with_res.sum().backward()

print("gradient magnitude reaching input, WITHOUT residuals:", x_no_res.grad.abs().mean().item())
print("gradient magnitude reaching input, WITH residuals:   ", x_with_res.grad.abs().mean().item())

gradient magnitude reaching input, WITHOUT residuals: 241908.453125
gradient magnitude reaching input, WITH residuals:    320.9870300292969


With a weight scale of 0.9 summed over 64 input dimensions per layer, the pre-activation variance actually *grows* with depth here rather than shrinking -- so this particular setup demonstrates an **exploding**, not vanishing, gradient without residuals (241,908 vs. 320 with residuals in the run above). Depending on the weight initialization scale, a deep stack can go either way -- vanish toward zero or blow up -- and both are real failure modes in practice. The point the numbers actually support: residual connections keep the gradient magnitude far more controlled either way, because the `+ x` term gives backpropagation a path that bypasses every `tanh` and every weight matrix entirely, regardless of how those weights happen to be scaled.

## 4. Pre-norm transformer sublayer pattern

In [5]:
class PreNormSublayer(nn.Module):
    """Wraps any sublayer (attention or feed-forward) in: x + sublayer(norm(x))"""
    def __init__(self, emb_dim, sublayer):
        super().__init__()
        self.norm = LayerNorm(emb_dim)
        self.sublayer = sublayer

    def forward(self, x):
        return x + self.sublayer(self.norm(x))

# demo with a placeholder sublayer (a simple linear layer stands in for attention/FFN here)
placeholder_sublayer = nn.Linear(768, 768)
wrapped = PreNormSublayer(768, placeholder_sublayer)

x = torch.randn(2, 5, 768)
out = wrapped(x)
print("input shape: ", x.shape)
print("output shape:", out.shape, " (unchanged -- this is what makes the block stackable)")

input shape:  torch.Size([2, 5, 768])
output shape: torch.Size([2, 5, 768])  (unchanged -- this is what makes the block stackable)


## Takeaway

LayerNorm keeps activation scale controlled at every layer, independent of batch size or sequence length. Residual connections keep the gradient from vanishing across depth by giving it a direct path around every sublayer. Together, they're what makes stacking 12+ transformer blocks trainable at all.

**Next notebook: Episode 7 — The feed-forward (MLP) block.**